In [1]:
import pandas as pd
import numpy as np
import pickle
import os

# Rutas
PROCESSED = '../data/processed/'

# Cargamos el pickle limpio de la fase 1
df = pd.read_pickle(PROCESSED + 'df_clean.pkl')

# Creamos una copia sobre la que trabajaremos
# NUNCA modificamos df directamente — es nuestra referencia limpia
df_features = df.copy()

pd.set_option('display.max_columns', None)

# Verificación
print("Shape:", df_features.shape)
print("Columnas:", df_features.columns.tolist())
print("\nPrimeras filas:")
df_features.head(3)

Shape: (3229, 12)
Columnas: ['budget', 'genres', 'keywords', 'original_language', 'overview', 'runtime', 'main_production_company', 'top_cast', 'director', 'release_year', 'release_month', 'success']

Primeras filas:


,budget,genres,keywords,original_language,overview,runtime,main_production_company,top_cast,director,release_year,release_month,success
0,237000000,"[Action, Adventure, Fantasy, Science Fiction]","[culture clash, future, space war, space colon...",en,"In the 22nd century, a paraplegic Marine is di...",162.0,Ingenious Film Partners,"[Sam Worthington, Zoe Saldana, Sigourney Weaver]",James Cameron,2009.0,12.0,1
1,300000000,"[Adventure, Fantasy, Action]","[ocean, drug abuse, exotic island, east india ...",en,"Captain Barbossa, long believed to be dead, ha...",169.0,Walt Disney Pictures,"[Johnny Depp, Orlando Bloom, Keira Knightley]",Gore Verbinski,2007.0,5.0,1
2,245000000,"[Action, Adventure, Crime]","[spy, based on novel, secret agent, sequel, mi...",en,A cryptic message from Bond’s past sends him o...,148.0,Columbia Pictures,"[Daniel Craig, Christoph Waltz, Léa Seydoux]",Sam Mendes,2015.0,10.0,1


In [2]:
# Tasa global de éxito del dataset — nuestro valor de referencia
global_rate = df_features['success'].mean()
m = 5  # parámetro de confianza

def director_weighted_rate(df, idx, m, global_rate):
    """
    Para una película (idx), calcula la tasa de éxito ponderada
    del director usando todas sus películas excepto la actual.
    """
    director = df.loc[idx, 'director']
    
    # Todas las películas del mismo director excepto la actual
    other_movies = df[(df['director'] == director) & (df.index != idx)]
    
    n = len(other_movies)          # cuántas películas tiene el director
    
    if n == 0:
        # Sin historial — usamos la media global
        return global_rate
    
    own_rate = other_movies['success'].mean()  # su tasa real
    
    # Fórmula de ponderación
    weighted_rate = (n * own_rate + m * global_rate) / (n + m)
    
    return weighted_rate

# Aplicamos la función a cada fila del dataframe
df_features['director_success_rate'] = [
    director_weighted_rate(df_features, idx, m, global_rate)
    for idx in df_features.index
]

# Verificación
print("Media global de referencia:", round(global_rate, 4))
print("\nEstadísticas de director_success_rate:")
print(df_features['director_success_rate'].describe().round(4))
print("\nEjemplos:")
print(df_features[['director', 'director_success_rate']].drop_duplicates('director').head(10))

Media global de referencia: 0.755

Estadísticas de director_success_rate:
count    3229.0000
mean        0.7720
std         0.0776
min         0.4619
25%         0.7550
50%         0.7550
75%         0.8250
max         0.9605
Name: director_success_rate, dtype: float64

Ejemplos:
            director  director_success_rate
0      James Cameron               0.888651
1     Gore Verbinski               0.706833
2         Sam Mendes               0.797742
3  Christopher Nolan               0.897930
4     Andrew Stanton               0.846895
5          Sam Raimi               0.814597
6       Byron Howard               0.755033
7        Joss Whedon               0.682166
8        David Yates               0.825023
9        Zack Snyder               0.888651


In [3]:
def actor_weighted_rate(df, idx, actor, m, global_rate):
    """
    Para un actor concreto en una película (idx), calcula su tasa
    de éxito ponderada usando todas sus apariciones excepto la actual.
    """
    # Todas las películas donde aparece este actor excepto la actual
    other_movies = df[
        (df['top_cast'].apply(lambda x: actor in x)) & 
        (df.index != idx)
    ]
    
    n = len(other_movies)
    
    if n == 0:
        return global_rate
    
    own_rate = other_movies['success'].mean()
    weighted_rate = (n * own_rate + m * global_rate) / (n + m)
    
    return weighted_rate

def cast_weighted_rate(df, idx, m, global_rate):
    """
    Promedia la tasa ponderada de los 3 actores del reparto.
    """
    actors = df.loc[idx, 'top_cast']
    
    if len(actors) == 0:
        return global_rate
    
    rates = [actor_weighted_rate(df, idx, actor, m, global_rate) 
             for actor in actors]
    
    return np.mean(rates)

# Aplicamos — esto puede tardar unos segundos
print("Calculando tasas del reparto...")
df_features['cast_success_rate'] = [
    cast_weighted_rate(df_features, idx, m, global_rate)
    for idx in df_features.index
]

# Verificación
print("Listo.\n")
print("Estadísticas de cast_success_rate:")
print(df_features['cast_success_rate'].describe().round(4))
print("\nEjemplos:")
print(df_features[['top_cast', 'cast_success_rate']].head(5))

Calculando tasas del reparto...
Listo.

Estadísticas de cast_success_rate:
count    3229.0000
mean        0.7605
std         0.0490
min         0.5532
25%         0.7307
50%         0.7611
75%         0.7925
max         0.9065
Name: cast_success_rate, dtype: float64

Ejemplos:
                                           top_cast  cast_success_rate
0  [Sam Worthington, Zoe Saldana, Sigourney Weaver]           0.721714
1     [Johnny Depp, Orlando Bloom, Keira Knightley]           0.748498
2      [Daniel Craig, Christoph Waltz, Léa Seydoux]           0.796303
3      [Christian Bale, Michael Caine, Gary Oldman]           0.719775
4    [Taylor Kitsch, Lynn Collins, Samantha Morton]           0.732941


In [4]:
def company_weighted_rate(df, idx, m, global_rate):
    """
    Para una película (idx), calcula la tasa de éxito ponderada
    de su productora usando todas sus películas excepto la actual.
    """
    company = df.loc[idx, 'main_production_company']
    
    # Si no hay productora registrada, devolvemos la media global
    if pd.isnull(company):
        return global_rate
    
    # Todas las películas de la misma productora excepto la actual
    other_movies = df[
        (df['main_production_company'] == company) & 
        (df.index != idx)
    ]
    
    n = len(other_movies)
    
    if n == 0:
        return global_rate
    
    own_rate = other_movies['success'].mean()
    weighted_rate = (n * own_rate + m * global_rate) / (n + m)
    
    return weighted_rate

# Aplicamos
print("Calculando tasas de productora...")
df_features['company_success_rate'] = [
    company_weighted_rate(df_features, idx, m, global_rate)
    for idx in df_features.index
]

# Verificación
print("Listo.\n")
print("Estadísticas de company_success_rate:")
print(df_features['company_success_rate'].describe().round(4))
print("\nEjemplos:")
print(df_features[['main_production_company', 'company_success_rate']].head(5))

Calculando tasas de productora...
Listo.

Estadísticas de company_success_rate:
count    3229.0000
mean        0.7888
std         0.0813
min         0.3146
25%         0.7550
50%         0.8119
75%         0.8469
max         0.9417
Name: company_success_rate, dtype: float64

Ejemplos:
   main_production_company  company_success_rate
0  Ingenious Film Partners              0.798817
1     Walt Disney Pictures              0.897752
2        Columbia Pictures              0.845736
3       Legendary Pictures              0.830272
4     Walt Disney Pictures              0.897752


In [5]:
# sklearn tiene una clase específica para esto: MultiLabelBinarizer
from sklearn.preprocessing import MultiLabelBinarizer

mlb = MultiLabelBinarizer()

# Ajustamos y transformamos la columna genres
# fit_transform aprende todos los géneros posibles y codifica simultáneamente
genres_encoded = mlb.fit_transform(df_features['genres'])

# Creamos un dataframe con los nombres de los géneros como columnas
genres_df = pd.DataFrame(
    genres_encoded,
    columns=[f'genre_{g.lower().replace(" ", "_")}' for g in mlb.classes_],
    index=df_features.index  # mantenemos el índice original para el merge
)

# Lo unimos a df_features
df_features = pd.concat([df_features, genres_df], axis=1)

# Verificación
print("Géneros codificados:", list(genres_df.columns))
print("\nShape tras encoding:", df_features.shape)
print("\nEjemplo — Avatar:")
print(df_features[genres_df.columns].iloc[0])

Géneros codificados: ['genre_action', 'genre_adventure', 'genre_animation', 'genre_comedy', 'genre_crime', 'genre_documentary', 'genre_drama', 'genre_family', 'genre_fantasy', 'genre_foreign', 'genre_history', 'genre_horror', 'genre_music', 'genre_mystery', 'genre_romance', 'genre_science_fiction', 'genre_thriller', 'genre_war', 'genre_western']

Shape tras encoding: (3229, 34)

Ejemplo — Avatar:
genre_action             1
genre_adventure          1
genre_animation          0
genre_comedy             0
genre_crime              0
genre_documentary        0
genre_drama              0
genre_family             0
genre_fantasy            1
genre_foreign            0
genre_history            0
genre_horror             0
genre_music              0
genre_mystery            0
genre_romance            0
genre_science_fiction    1
genre_thriller           0
genre_war                0
genre_western            0
Name: 0, dtype: int64


In [6]:
df_features.drop(columns=['genre_foreign'], inplace=True)

print("Shape tras eliminar genre_foreign:", df_features.shape)

Shape tras eliminar genre_foreign: (3229, 33)


In [7]:
def assign_season(month):
    if month in [6, 7, 8]:
        return 'summer'
    elif month in [11, 12]:
        return 'christmas'
    elif month in [3, 4, 5]:
        return 'spring'
    else:
        return 'off_season'

# Creamos la columna de temporada
df_features['release_season'] = df_features['release_month'].apply(assign_season)

# One-hot encoding de la temporada
# drop_first=False porque queremos todas las categorías visibles
season_dummies = pd.get_dummies(
    df_features['release_season'],
    prefix='season',
    dtype=int
)

df_features = pd.concat([df_features, season_dummies], axis=1)

# Verificación
print("Temporadas codificadas:", list(season_dummies.columns))
print("\nDistribución de temporadas:")
print(df_features['release_season'].value_counts())
print("\nTasa de éxito por temporada:")
print(df_features.groupby('release_season')['success'].mean().round(3))

Temporadas codificadas: ['season_christmas', 'season_off_season', 'season_spring', 'season_summer']

Distribución de temporadas:
release_season
off_season    1097
summer         837
spring         704
christmas      591
Name: count, dtype: int64

Tasa de éxito por temporada:
release_season
christmas     0.799
off_season    0.690
spring        0.768
summer        0.798
Name: success, dtype: float64


In [8]:
from sklearn.preprocessing import StandardScaler

# Columnas numéricas continuas que necesitan escalado
# release_year lo incluimos aquí también — sus valores (1916-2016) 
# están en una escala muy distinta al resto de features binarios
cols_to_scale = ['budget', 'runtime', 'release_year',
                 'director_success_rate', 'cast_success_rate', 
                 'company_success_rate']

# Guardamos la lista para usarla en fase 3
print("Columnas que se escalarán en fase 3:")
for col in cols_to_scale:
    print(f"  {col}: min={df_features[col].min():.2f}, max={df_features[col].max():.2f}")

Columnas que se escalarán en fase 3:
  budget: min=1.00, max=380000000.00
  runtime: min=41.00, max=338.00
  release_year: min=1916.00, max=2016.00
  director_success_rate: min=0.46, max=0.96
  cast_success_rate: min=0.55, max=0.91
  company_success_rate: min=0.31, max=0.94


### Eliminamos original_language 
 
En el EDA concluimos que el 96% del dataset es en inglés. Eso significa que esta variable tiene varianza casi nula — casi todas las filas tienen el mismo valor. Una variable con varianza casi nula no aporta señal al modelo porque no discrimina entre películas — casi todas son iguales en ese feature.
Además, si hacemos one-hot encoding de los idiomas, tendríamos una columna language_en con 96% de unos y el resto de columnas con menos del 1% cada una — columnas prácticamente vacías que solo añaden ruido y dimensionalidad innecesaria.

In [9]:
# Eliminamos original_language
df_features.drop(columns=['original_language'], inplace=True)

# Verificación
print("Shape:", df_features.shape)
print("Columnas:", df_features.columns.tolist())

Shape: (3229, 37)
Columnas: ['budget', 'genres', 'keywords', 'overview', 'runtime', 'main_production_company', 'top_cast', 'director', 'release_year', 'release_month', 'success', 'director_success_rate', 'cast_success_rate', 'company_success_rate', 'genre_action', 'genre_adventure', 'genre_animation', 'genre_comedy', 'genre_crime', 'genre_documentary', 'genre_drama', 'genre_family', 'genre_fantasy', 'genre_history', 'genre_horror', 'genre_music', 'genre_mystery', 'genre_romance', 'genre_science_fiction', 'genre_thriller', 'genre_war', 'genre_western', 'release_season', 'season_christmas', 'season_off_season', 'season_spring', 'season_summer']


In [10]:
# Eliminamos columnas ya procesadas o reservadas para NLP futuro
cols_to_drop = ['genres', 'keywords', 'overview', 'main_production_company',
                'top_cast', 'director', 'release_month', 'release_season']

df_features.drop(columns=cols_to_drop, inplace=True)

# Verificación final
print("Shape:", df_features.shape)
print("\nColumnas finales:")
for col in df_features.columns.tolist():
    print(f"  {col}")

Shape: (3229, 29)

Columnas finales:
  budget
  runtime
  release_year
  success
  director_success_rate
  cast_success_rate
  company_success_rate
  genre_action
  genre_adventure
  genre_animation
  genre_comedy
  genre_crime
  genre_documentary
  genre_drama
  genre_family
  genre_fantasy
  genre_history
  genre_horror
  genre_music
  genre_mystery
  genre_romance
  genre_science_fiction
  genre_thriller
  genre_war
  genre_western
  season_christmas
  season_off_season
  season_spring
  season_summer


In [11]:
# Guardamos df_features
df_features.to_pickle(PROCESSED + 'df_features.pkl')

# Verificación
df_check = pd.read_pickle(PROCESSED + 'df_features.pkl')
print("Shape original:", df_features.shape)
print("Shape desde pickle:", df_check.shape)
print("Idénticos:", df_features.equals(df_check))

Shape original: (3229, 29)
Shape desde pickle: (3229, 29)
Idénticos: True
